In [2]:
import mediapipe as mp
import cv2
import playsound
from threading import Thread
import time
import numpy as np
from collections import deque


In [3]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

ALARM = "../alarm-clock-90867.mp3"


In [12]:
def sound_alarm(path=ALARM):
    playsound.playsound(path)
    

# # 1. Cabeça projetada para frente (principal indicador)
# def desvioZ_suavizado(desvioZ, history, window=45):
#     history.append(smoth_pitch)
#     if len(history) > window:
#         history.pop(0)
#     return np.mean(history)

def Cabeca_projetada_para_frente(landmarks):
    nose     = landmarks[mp_holistic.PoseLandmark.NOSE]
    left_sh  = landmarks[mp_holistic.PoseLandmark.LEFT_SHOULDER]
    right_sh = landmarks[mp_holistic.PoseLandmark.RIGHT_SHOULDER]

    shoulder_mid_z = (left_sh.z + right_sh.z) / 2.0

    # diferença em profundidade
    diff_z = shoulder_mid_z - nose.z
    # para divisao por 0 não acontecer

     # régua 2D da pessoa
    shoulder_width = abs(left_sh.x - right_sh.x)   # abs(-5 )= 5, abs(-0.0001)= 0.0001 
    if shoulder_width < 0.001:
        return None

  
    diff = diff_z / shoulder_width  # normalizar a diferença para funcionar independente da distância da câmera.
   
    return diff


# 2. Inclinação da cabeça (pescoço torto)
def Inclinacao_cabeca(landmarks):
    
    left_ear = landmarks[mp_holistic.PoseLandmark.LEFT_EAR]
    right_ear = landmarks[mp_holistic.PoseLandmark.RIGHT_EAR]
    
    DiferencaEarAltura = abs(left_ear.y - right_ear.y)
    inclinada= False
    if DiferencaEarAltura > 0.05:  
        inclinada = True
    return {"diferenca": DiferencaEarAltura,  "inclinacao": inclinada}




In [13]:
FORWARD_HISTORY = []
FORWARD_HEAD_THRESH = 1.75
smooth_pitch = 0
cap = cv2.VideoCapture(0)
# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic: 
    # poderia ser usado uma variavel (holistic.close() necessario), mas é usado assim devido liberação de memória autoática ...
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Make Detections
        results = holistic.process(image)
        
        if results.pose_landmarks is None:
            print("No pose landmarks detected.")
        else:
            landmarks = results.pose_landmarks.landmark

    #  # 1. Cabeça projetada para frente (principal indicador)
    #         desvioZ = Cabeca_projetada_para_frente(landmarks)
    #         if desvioZ is not None:
    #             smooth_pitch = 0.85 * smooth_pitch + (1 - 0.85) * desvioZ
    #             # desvioZ_suave = desvioZ_suavizado(desvioZ, FORWARD_HISTORY) desvioZ_suave
    #             print(smooth_pitch)
    #             if smooth_pitch > FORWARD_HEAD_THRESH:
    #                 print('True', smooth_pitch)
    #             else:
    #                 print('False', smooth_pitch)

                    
    #  2. Inclinação da cabeça (pescoço torto)
            cabecaInclinada = Inclinacao_cabeca(landmarks)
            
            print(cabecaInclinada["inclinacao"])
  
 




            # Recolor image back to BGR for rendering
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
          
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
                                    mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                    mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2)
                                    )

        # mostrando na tela
        cv2.namedWindow('Raw Webcam Feed', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Raw Webcam Feed', 1100, 800)
        cv2.imshow('Raw Webcam Feed', cv2.flip(image,1))
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
            
cap.release()
cv2.destroyAllWindows()


False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False


In [22]:
# fechar camera/ janelas
cap.release()
cv2.destroyAllWindows()

In [2]:



import cv2
import numpy as np
import mediapipe as mp

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# ==============================
# CONFIG
# ==============================
ALPHA = 0.95
THRESH = 0.06

smooth_score = 0

# ==============================
# FUNÇÃO DE POSTURA
def posture_score(lm):

    nose = lm[mp_holistic.PoseLandmark.NOSE]
    left_sh = lm[mp_holistic.PoseLandmark.LEFT_SHOULDER]
    right_sh = lm[mp_holistic.PoseLandmark.RIGHT_SHOULDER]

    # centro dos ombros
    shoulder_y = (left_sh.y + right_sh.y) / 2
    shoulder_z = (left_sh.z + right_sh.z) / 2

    # escala do corpo (normalização)
    scale = abs(left_sh.x - right_sh.x)

    if scale < 1e-6:
        return None

    # ==========================
    # COMPONENTE 1: Y (principal)
    # ==========================
    pitch_y = (nose.y - shoulder_y) / scale

    # ==========================
    # COMPONENTE 2: Z (auxiliar)
    # ==========================
    depth_z = (shoulder_z - nose.z) / scale

    # ==========================
    # SCORE FINAL (junção)
    # ==========================
    score = pitch_y + 0.5 * depth_z

    return score


# ==============================
# WEBCAM
cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)

        if results.pose_landmarks:

            lm = results.pose_landmarks.landmark

            score = posture_score(lm)

            if score is not None:

                # ==========================
                # SUAVIZAÇÃO (EMA)
                # ==========================
                smooth_score = ALPHA * smooth_score + (1 - ALPHA) * score

                # ==========================
                # DECISÃO
                # ==========================
                if smooth_score > THRESH:
                    status = "CABECA PRA FRENTE"
                    color = (0, 0, 255)
                else:
                    status = "POSTURA OK"
                    color = (0, 255, 0)

                # ==========================
                # HUD
                # ==========================
                cv2.putText(
                    frame,
                    f"Score: {smooth_score:.3f}",
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2
                )

                cv2.putText(
                    frame,
                    status,
                    (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2
                )

        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_holistic.POSE_CONNECTIONS
        )

        cv2.imshow("Postura - Y + Z Fusion", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# calcular baseline


def calcular_PosturaPAdrao(landmarks,valores):
    # duração da coleta (em segundos)

    nose = landmarks[mp_holistic.PoseLandmark.NOSE]
    left = landmarks[mp_holistic.PoseLandmark.LEFT_SHOULDER]
    right = landmarks[mp_holistic.PoseLandmark.RIGHT_SHOULDER]

    # centro dos ombros
    shoulder_z = (left.z + right.z) / 2

    # diferença
    diff = shoulder_z - nose.z
    print("Diff:", diff)
    valores.append(diff)


    
CALIBRATION_TIME = 10
valores = []
start_time = time.time()

cap = cv2.VideoCapture(0)
# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic: 
    # poderia ser usado uma variavel (holistic.close() necessario), mas é usado assim devido liberação de memória autoática ...
    
    while cap.isOpened() and  time.time() - start_time < CALIBRATION_TIME:
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Make Detections
        results = holistic.process(image)
        
        if results.pose_landmarks is None:
            print("No pose landmarks detected.")
        else:
            landmarks = results.pose_landmarks.landmark

     # 1. Cabeça projetada para frente (principal indicador)
            calcular_PosturaPAdrao(landmarks,valores)
            
            # Recolor image back to BGR for rendering
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            # # 4. Pose Detections
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
                                    mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                    mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2)
                                    )

        # mostrando na tela
        cv2.namedWindow('Raw Webcam Feed', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Raw Webcam Feed', 1100, 800)
        cv2.imshow('Raw Webcam Feed', cv2.flip(image,1))
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    baseline = sum(valores) / len(valores)
    print("---------------concluída! -------------------")
    print(f'minVal: {min(valores)}, maxVal: {max(valores)}')
    print("PosturaPadrao:", baseline)
    print("Amostras:", len(valores))
            
cap.release()
cv2.destroyAllWindows()
